# SEAS5

In [1]:
%load_ext jupyter_black
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd

from src.datasources import seas5, cerf, emdat
from src.utils import db_utils

In [3]:
def calculate_rp(group, col_names, ascending: bool = False):
    for col_name in col_names:
        group[f"rank_{col_name}"] = group[col_name].rank(ascending=False)
        group[f"rp_{col_name}"] = (len(group) + 1) / group[f"rank_{col_name}"]
    return group

In [74]:
df_emdat = emdat.load_emdat()

In [126]:
df_emdat_year = (
    df_emdat.groupby("Start Year")["Total Affected"].sum().reset_index()
)
full_year_range = range(2000, df_emdat_year["Start Year"].max() + 1)
df_emdat_year = (
    df_emdat_year.set_index("Start Year")
    .reindex(full_year_range)
    .reset_index()
    .fillna(0)
    .astype(int)
    .rename(columns={"Start Year": "year", "Total Affected": "total_affected"})
)

In [127]:
df_emdat_year

,year,total_affected
0,2000,0
1,2001,0
2,2002,0
3,2003,12120
4,2004,0
5,2005,0
6,2006,25610
7,2007,121043
8,2008,4870
9,2009,151500


In [128]:
df_emdat_year = calculate_rp(df_emdat_year, col_name="total_affected")

In [129]:
df_emdat_year["TP_3yr"] = df_emdat_year["rp_total_affected"] > 3
df_emdat_year["TP_5yr"] = df_emdat_year["rp_total_affected"] > 5
df_emdat_year

,year,total_affected,rank_total_affected,rp_total_affected,TP_3yr,TP_5yr
0,2000,0,19.5,1.333333,False,False
1,2001,0,19.5,1.333333,False,False
2,2002,0,19.5,1.333333,False,False
3,2003,12120,9.0,2.888889,False,False
4,2004,0,19.5,1.333333,False,False
5,2005,0,19.5,1.333333,False,False
6,2006,25610,7.0,3.714286,True,False
7,2007,121043,4.0,6.500000,True,True
8,2008,4870,12.0,2.166667,False,False
9,2009,151500,1.0,26.000000,True,True


In [25]:
df_cerf = cerf.load_cerf()

In [26]:
df_cerf

,year,month,ADM_PCODE,admin_level
0,2009,9,BF130004,3
1,2010,7,BF52,1
2,2010,7,BF56,1
3,2010,7,BF54,1
4,2010,7,BF49,1
5,2010,7,BF55,1


In [4]:
df_seas5 = seas5.load_seas5()

In [5]:
df_seas5

,iso3,pcode,valid_date,issued_date,leadtime,adm_level,mean,median,min,max,count,sum,std
0,BFA,BF,1990-01-01,1990-01-01,0,0,0.000075,-2.773725e-20,-2.773725e-20,0.004727,9045,0.682178,0.000297
1,BFA,BF,1990-02-01,1990-02-01,0,0,0.063994,4.244596e-02,3.344379e-03,0.387923,9045,578.824000,0.071019
2,BFA,BF,1990-02-01,1990-01-01,1,0,0.048439,3.064263e-02,6.175856e-03,0.269024,9045,438.130830,0.047557
3,BFA,BF,1990-03-01,1990-03-01,0,0,0.101743,6.558001e-02,1.131557e-03,0.723492,9045,920.262400,0.112739
4,BFA,BF,1990-03-01,1990-02-01,1,0,0.157934,1.057125e-01,4.500976e-03,0.918748,9045,1428.508500,0.161776
...,...,...,...,...,...,...,...,...,...,...,...,...,...
218885,BFA,BF5201,2025-08-01,2025-02-01,6,2,6.327063,6.530020e+00,5.590717e+00,7.164506,312,1974.043600,0.311466
218886,BFA,BF4903,2025-08-01,2025-02-01,6,2,6.029475,5.945247e+00,5.354002e+00,7.246327,312,1881.196200,0.418329
218887,BFA,BF5001,2025-08-01,2025-02-01,6,2,7.408233,7.257753e+00,7.107964e+00,7.816766,140,1037.152600,0.286134
218888,BFA,BF4605,2025-08-01,2025-02-01,6,2,7.733387,7.796920e+00,7.393615e+00,7.845301,124,958.939940,0.114857


In [43]:
adm_level = 0

In [253]:
df_seas5_adm0 = (
    df_seas5[df_seas5["adm_level"] == 0]
    .groupby(
        [
            df_seas5["valid_date"].dt.month.rename("valid_month"),
            df_seas5["issued_date"].dt.month.rename("issued_month"),
        ]
    )
    .apply(calculate_rp, col_names=["mean", "max"])
    .reset_index()
    .drop(columns="level_2")
)

In [254]:
df_compare = df_emdat_year.copy()
df_compare["cerf"] = df_compare["year"].isin(df_cerf["year"].unique())

In [255]:
def display_metrics_df(df, bar_col: str = "total_affected"):
    def highlight_true(val, color: str = "crimson"):
        if isinstance(val, bool) and val is True:
            return f"background-color: {color}"
        return ""

    cols = [x for x in df.columns if x != bar_col] + [bar_col]
    df = df[cols].sort_values(bar_col, ascending=False)
    display(
        df[cols]
        .style.bar(
            subset=bar_col,
            color="coral",
            props="width: 300px;",
        )
        .map(highlight_true)
        .map(highlight_true, color="dodgerblue", subset="cerf")
        .set_table_styles(
            {
                bar_col: [
                    {"selector": "th", "props": [("text-align", "left")]},
                    {"selector": "td", "props": [("text-align", "left")]},
                ]
            }
        )
        .format({bar_col: "{:,}"})
    )

In [256]:
def get_thresh_name(
    lt: int = 0, period: str = "mo", val_col: str = "mean", rp: int = 3
):
    return f"lt{lt}_{period}_{val_col}_{rp}rp"

In [264]:
months = [6, 7, 8, 9]
n_years_total = df_compare["year"].nunique()

rp_threshs = [3, 5, 8]
lt = 0

for val_col in ["mean", "max"]:
    for rp_thresh in rp_threshs:
        trigger_name = get_thresh_name(
            lt=lt, period="mo", val_col=val_col, rp=rp_thresh
        )
        df_triggers = df_seas5_adm0[
            (df_seas5_adm0[f"rp_{val_col}"] > rp_thresh)
            & (df_seas5_adm0["issued_month"].isin(months))
            & (df_seas5_adm0["leadtime"] == 0)
        ]
        years_triggered = df_triggers["issued_date"].dt.year.unique()
        df_compare[trigger_name] = df_compare["year"].isin(years_triggered)

In [267]:
cols = [x for x in df_compare.columns if "8rp" in x] + [
    "total_affected",
    "cerf",
]
display_metrics_df(df_compare.set_index("year")[cols])

,lt0_mo_mean_8rp,lt0_mo_max_8rp,cerf,total_affected
year,,,,
2009,False,True,True,"151,500"
2010,True,True,True,"133,362"
2020,True,False,False,"130,452"
2007,False,False,False,"121,043"
2016,False,False,False,"34,893"
2015,True,True,False,"28,925"
2006,False,False,False,"25,610"
2012,False,False,False,"21,000"
2003,True,True,False,"12,120"


In [268]:
n_years_total

25

In [272]:
tp_col = "TP_3yr"

dicts = []
for val_col in ["mean", "max"]:
    for rp in rp_threshs:
        trigger_name = get_thresh_name(rp=rp, val_col=val_col)
        dicts.append(
            {
                "val_col": val_col,
                "rp_trig": rp,
                "rp_eff": n_years_total / df_compare[trigger_name].sum(),
                "TPR": df_compare[[tp_col, trigger_name]].all(axis=1).sum()
                / df_compare[tp_col].sum(),
                "PPV": df_compare[[tp_col, trigger_name]].all(axis=1).sum()
                / df_compare[trigger_name].sum(),
            }
        )

df_metrics = pd.DataFrame(dicts)

In [273]:
df_metrics

,val_col,rp_trig,rp_eff,TPR,PPV
0,mean,3,1.562500,0.625,0.312500
1,mean,5,2.500000,0.500,0.400000
2,mean,8,5.000000,0.375,0.600000
3,max,3,1.315789,0.750,0.315789
4,max,5,2.083333,0.375,0.250000
5,max,8,3.125000,0.375,0.375000
